# Vital Signs EDA: 3-Variable Analysis (HR, SBP, DBP)

Focused analysis on the 3 cardiovascular vitals selected for LSTM modeling:
- **Heart Rate** (LOINC 8867-4)
- **Systolic Blood Pressure** (LOINC 8480-6)
- **Diastolic Blood Pressure** (LOINC 8462-4)

**Rationale for dropping other vitals:**
- **Temperature** — missing ~90% of timestamps, MCAR rejected (see 5v EDA)
- **Respiratory Rate** — observation-recorded, noisy signal
- **SpO2** — skewed distribution, confounded by unmeasured conditions

**No missingness section** — SBP/DBP co-occur from the same BP measurement; formal MCAR testing is not meaningful for 3 cardiovascular vitals.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch

sys.path.insert(0, str(Path.cwd().parent))

from src.eda.queries import load_eda_table
from src.eda.vitals import (
    VITALS_3,
    connect_vitals,
    filter_complete,
    plot_completeness_bar,
    plot_delta_histogram,
)

conn = connect_vitals()
df = load_eda_table(conn)
print(f"Loaded {len(df):,} timestamps, {df.encounter_id.nunique():,} encounters")

## 2. Data Overview

In [ ]:
print("Column types:")
display(df.dtypes.to_frame("dtype"))

print(f"\nTotal timestamps: {len(df):,}")
print(f"Unique patients: {df.patient_id.nunique():,}")
print(f"Unique encounters: {df.encounter_id.nunique():,}")

first_enc = df.encounter_id.iloc[0]
display(df[df.encounter_id == first_enc].head(12))

## 3. Timestamp Completeness (3 Vitals)

In [ ]:
v3_rows = df[df["n_vitals_3"] >= 1]
vc = v3_rows["n_vitals_3"].value_counts().sort_index()
completeness = pd.DataFrame({
    "num_vitals": vc.index,
    "num_timestamps": vc.values,
    "pct": (vc.values / vc.sum() * 100).round(2),
})
display(completeness)

fig, ax = plt.subplots(figsize=(10, 5))
plot_completeness_bar(completeness, ax, n_vitals=3)
plt.tight_layout()
plt.show()

## 4. Time Delta Analysis

In [ ]:
complete_3v = filter_complete(df, "n_vitals_3", 3)
deltas_3v = complete_3v["delta_min"].dropna()

# Percentile summary
delta_stats = pd.DataFrame([{
    "n_deltas": len(deltas_3v),
    "min": deltas_3v.min().round(1),
    "p10": deltas_3v.quantile(0.10).round(1),
    "p25": deltas_3v.quantile(0.25).round(1),
    "p50_median": deltas_3v.quantile(0.50).round(1),
    "p75": deltas_3v.quantile(0.75).round(1),
    "p90": deltas_3v.quantile(0.90).round(1),
    "p95": deltas_3v.quantile(0.95).round(1),
    "max": deltas_3v.max().round(1),
}])
display(delta_stats)

# Bucketed histogram
bins = [0, 60, 120, 180, 240, 360, 480, float("inf")]
labels = ["0-1 hr", "1-2 hrs", "2-3 hrs", "3-4 hrs", "4-6 hrs", "6-8 hrs", ">8 hrs"]
cut = pd.cut(deltas_3v, bins=bins, labels=labels, right=True)
bucket_counts = cut.value_counts().reindex(labels)
buckets = pd.DataFrame({
    "time_bucket": labels,
    "count": bucket_counts.values,
    "pct": (bucket_counts.values / bucket_counts.sum() * 100).round(2),
})

fig, ax = plt.subplots(figsize=(12, 6))
plot_delta_histogram(buckets, "Distribution of Time Deltas (3 Vitals: HR, SBP, DBP)", ax)
plt.tight_layout()
plt.show()

In [ ]:
# Average delta by measurement position
delta_pos = complete_3v[complete_3v["delta_min"].notna()].copy()
delta_pos_stats = delta_pos.groupby("obs_position")["delta_min"].agg(
    avg_delta_min="mean",
    median_delta_min="median",
    n_encounters="count",
).round(1).reset_index()
delta_pos_stats.columns = ["delta_idx", "avg_delta_min", "median_delta_min", "n_encounters"]
plot_data = delta_pos_stats[delta_pos_stats["n_encounters"] >= 50].copy()

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(plot_data["delta_idx"], plot_data["avg_delta_min"], "o-", color="steelblue", linewidth=2, markersize=8, label="Mean")
ax1.plot(plot_data["delta_idx"], plot_data["median_delta_min"], "s-", color="coral", linewidth=2, markersize=8, label="Median")
ax1.set_xlabel("Delta Position (1st, 2nd, 3rd, ...)", fontsize=12)
ax1.set_ylabel("Delta (minutes)", fontsize=12)
ax1.set_title("Time Delta by Measurement Position (3 Vitals)", fontsize=14, fontweight="bold")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.bar(plot_data["delta_idx"], plot_data["n_encounters"], alpha=0.2, color="gray")
ax2.set_ylabel("Number of Encounters", fontsize=12, color="gray")
ax2.tick_params(axis="y", labelcolor="gray")
plt.tight_layout()
plt.show()

## 5. Sequence Length & Coverage

In [ ]:
# Sequence length summary
all_encounters = df["encounter_id"].unique()
enc_3v_counts = complete_3v.groupby("encounter_id").size()
enc_seq_3v = pd.Series(0, index=all_encounters).add(enc_3v_counts, fill_value=0).astype(int)

n_enc = len(all_encounters)
seq_lengths = pd.DataFrame([{
    "n_encounters": n_enc,
    "encounters_usable": int((enc_seq_3v >= 2).sum()),
    "pct_usable": round(100 * (enc_seq_3v >= 2).sum() / n_enc, 1),
    "p25": enc_seq_3v.quantile(0.25).round(0),
    "p50": enc_seq_3v.quantile(0.50).round(0),
    "p75": enc_seq_3v.quantile(0.75).round(0),
    "p95": enc_seq_3v.quantile(0.95).round(0),
}])
display(seq_lengths)

# Histogram of sequence lengths
def _seq_bucket(n):
    if n == 0: return "0"
    if n == 1: return "1"
    if n == 2: return "2"
    if n == 3: return "3"
    if n <= 5: return "4-5"
    if n <= 10: return "6-10"
    return ">10"

bucket_order = ["0", "1", "2", "3", "4-5", "6-10", ">10"]
seq_bucketed = enc_seq_3v.map(_seq_bucket)
seq_counts = seq_bucketed.value_counts().reindex(bucket_order, fill_value=0)
seq_hist = pd.DataFrame({
    "seq_length": seq_counts.index,
    "num_encounters": seq_counts.values,
    "pct": (seq_counts.values / seq_counts.sum() * 100).round(2),
})

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#ff6b6b" if x in ["0", "1"] else "#6bcb77" for x in seq_hist["seq_length"]]
bars = ax.bar(seq_hist["seq_length"], seq_hist["pct"], color=colors, edgecolor="black")
ax.set_xlabel("Number of Complete Timestamps per Encounter (3 Vitals)", fontsize=12)
ax.set_ylabel("Percentage of Encounters (%)", fontsize=12)
ax.set_title("LSTM Sequence Length Distribution (3 Vitals)", fontsize=14, fontweight="bold")
for bar, pct in zip(bars, seq_hist["pct"]):
    ax.annotate(f"{pct}%", xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=10)
ax.legend(handles=[
    Patch(facecolor="#ff6b6b", edgecolor="black", label="Unusable for LSTM (0-1)"),
    Patch(facecolor="#6bcb77", edgecolor="black", label="Usable (2+)"),
], loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Timestep coverage
thresholds = [60, 120, 180, 240, 360, 480]
threshold_labels = ["1 hour", "2 hours", "3 hours", "4 hours", "6 hours", "8 hours"]
total_deltas = len(deltas_3v)
coverage_data = []
for mins, label in zip(thresholds, threshold_labels):
    pct = round(100 * (deltas_3v <= mins).sum() / total_deltas, 1)
    coverage_data.append({"timestep_min": mins, "timestep_label": label, "pct_covered": pct})
coverage = pd.DataFrame(coverage_data)
coverage["gaps_to_fill"] = 100 - coverage["pct_covered"]

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(coverage))
ax.bar(x, coverage["pct_covered"], color="steelblue", edgecolor="black", alpha=0.8)
ax.bar(x, coverage["gaps_to_fill"], bottom=coverage["pct_covered"], color="lightcoral", edgecolor="black", alpha=0.6)
ax.set_xticks(list(x))
ax.set_xticklabels(coverage["timestep_label"], fontsize=11)
ax.set_xlabel("Fixed Timestep Choice", fontsize=12)
ax.set_ylabel("Percentage (%)", fontsize=12)
ax.set_title("Timestep Coverage: % of Deltas Within Timestep (3 Vitals)", fontsize=14, fontweight="bold")
ax.axhline(y=80, color="green", linestyle="--", linewidth=2, label="80% target")
for i, covered in enumerate(coverage["pct_covered"]):
    ax.annotate(f"{covered}%", xy=(i, covered / 2), ha="center", va="center", fontsize=10, fontweight="bold", color="white")
ax.legend(handles=[
    Patch(facecolor="steelblue", edgecolor="black", label="Covered (no interpolation)"),
    Patch(facecolor="lightcoral", edgecolor="black", label="Gaps (need interpolation)"),
], loc="lower right")
plt.tight_layout()
plt.show()

## 6. Sequence Length & Delta.

In [ ]:
# Relationship between sequence length and average time delta
enc_avg_delta = complete_3v[complete_3v["delta_min"].notna()].groupby("encounter_id")["delta_min"].mean()
enc_n_ts = complete_3v.groupby("encounter_id").size()

seq_delta = pd.DataFrame({
    "n_timesteps": enc_n_ts,
    "avg_delta": enc_avg_delta,
}).dropna()

seq_delta_stats = seq_delta.groupby("n_timesteps").agg(
    mean_avg_delta=("avg_delta", "mean"),
    std_avg_delta=("avg_delta", "std"),
    n_encounters=("avg_delta", "count"),
).round(1).reset_index()

plot_data = seq_delta_stats[seq_delta_stats["n_encounters"] >= 10].copy()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(plot_data["n_timesteps"], plot_data["mean_avg_delta"],
        "o-", color="steelblue", linewidth=2, markersize=6, label="Mean Avg Delta")
ax.fill_between(
    plot_data["n_timesteps"],
    plot_data["mean_avg_delta"] - plot_data["std_avg_delta"],
    plot_data["mean_avg_delta"] + plot_data["std_avg_delta"],
    alpha=0.25, color="steelblue", label="\u00b1 1 Std Dev",
)
ax.set_xlabel("Number of Timesteps per Encounter", fontsize=12)
ax.set_ylabel("Average Delta (minutes)", fontsize=12)
ax.set_title("Average Time Delta vs. Sequence Length (3 Vitals)",
             fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Sequence length histogram + reverse cumulative
enc_dist = enc_seq_3v[enc_seq_3v >= 1]
seq_dist = enc_dist.value_counts().sort_index().reset_index()
seq_dist.columns = ["n_timesteps", "n_encounters"]

# Reverse cumulative: encounters with >= n timesteps
seq_dist["reverse_cumulative"] = seq_dist["n_encounters"][::-1].cumsum()[::-1]

fig, ax1 = plt.subplots(figsize=(12, 6))

# Histogram bars
ax1.bar(seq_dist["n_timesteps"], seq_dist["n_encounters"],
        color="steelblue", edgecolor="black", alpha=0.8, label="Frequency")
ax1.set_xlabel("Number of Timesteps per Encounter", fontsize=12)
ax1.set_ylabel("Number of Encounters", fontsize=12)
ax1.set_title("Sequence Length Distribution (3 Vitals)",
              fontsize=14, fontweight="bold")

# Reverse cumulative on twin axis
ax2 = ax1.twinx()
ax2.plot(seq_dist["n_timesteps"], seq_dist["reverse_cumulative"],
         color="red", marker="o", linewidth=2, markersize=5, label="\u2265 N timesteps")
ax2.set_ylabel("Encounters with \u2265 N Timesteps", fontsize=12, color="red")
ax2.tick_params(axis="y", labelcolor="red")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Summary

In [ ]:
n_vitals = len(VITALS_3)
pct_usable = seq_lengths["pct_usable"].iloc[0]
median_delta = delta_stats["p50_median"].iloc[0]

print(f"""
KEY FINDINGS ({n_vitals}-Vital Analysis: HR, SBP, DBP)
{'=' * 50}

1. COMPLETENESS
   - {n_vitals} cardiovascular vitals co-occur frequently
   - SBP & DBP come from the same BP measurement

2. TIME DELTAS
   - Median delta: ~{median_delta:.0f} minutes between complete timestamps
   - More usable timestamps than the 5v analysis

3. SEQUENCE LENGTHS
   - ~{pct_usable:.0f}% of encounters have 2+ complete timestamps (usable for LSTM)

4. TIMESTEP RECOMMENDATION
   - Check coverage chart above for optimal timestep choice
""")

conn.close()
print("Connection closed.")